# Lab: Hands-on with Torchtext & Pre-trained Embeddings

## 1. Introduction to Torchtext

PyTorch provides a library called **torchtext** to handle Natural Language Processing (NLP) data.

While earlier versions focused on data loading iterators (Field, BucketIterator), the modern API focuses on building blocks:
1.  **Vocab**: Managing the mapping between words (strings) and indices (integers).
2.  **Vectors**: Loading pre-trained embeddings like GloVe and FastText.
3.  **Tokenizers**: Splitting text into tokens.

In this lab, we will explore:
*   How to build a vocabulary.
*   How to load massive pre-trained vectors.
*   How to compare words using Cosine Similarity.
*   The difference between Count-based (GloVe) and Prediction-based (FastText) embeddings.



In [1]:
import torch
import torchtext

from torchtext.data.utils import get_tokenizer
from collections import Counter
from torchtext.vocab import Vocab, build_vocab_from_iterator

print(f"Torch version: {torch.__version__}")
print(f"TorchText version: {torchtext.__version__}")



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Python312\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Python312\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Python312\Lib\site-packages\ipykernel\kernelapp.py", line 758, in start
    self.io_loop.start()
  File "C:\Python312\Lib\site-packages\tornado\platform\asyncio.py", line 211, in 

Torch version: 2.2.2+cpu
TorchText version: 0.17.2+cpu


## 2. Tokenization and Vocabulary Building

Before using embeddings, we need to process raw text.
1.  **Tokenization**: Breaking text into words/subwords.
2.  **Vocabulary**: Assigning a unique ID to each unique token.



In [2]:
# 1. Tokenizer
tokenizer = get_tokenizer('basic_english')

text_sample = "we are playing with torchtext libraries and their methods."
tokens = tokenizer(text_sample)
print(f"Tokens: {tokens}")

# 2. Build Vocabulary
# In a real scenario, this iterator would come from a dataset
def yield_tokens(data_iter):
    for text in data_iter:
        yield tokenizer(text)

corpus = [
    "The king is a man.",
    "The queen is a woman.",
    "Apple and orange are fruits.",
    "The sun is hot."
]

# Create vocab
vocab = build_vocab_from_iterator(yield_tokens(corpus), specials=["<unk>"])
vocab.set_default_index(vocab["<unk>"])

print(f"Index for 'king': {vocab['king']}")
print(f"Index for 'unknown_word': {vocab['unknown_word']}")
print(f"Indices for first sentence: {[vocab[token] for token in tokenizer(corpus[0])]}")


NameError: name 'get_tokenizer' is not defined

## 3. Loading Pre-trained Embeddings (GloVe)

Training your own embeddings requires massive datasets (Wikipedia size).

Instead, we use Pre-trained Embeddings.
**GloVe (Global Vectors)** is a popular choice. 
It learns vectors such that their dot product equals the logarithm of the words' probability of co-occurrence.

*Note: The first time you run this, it will download ~800MB (GloVe 6B).*



In [ ]:
# Load GloVe vectors
# name='6B' means trained on 6 Billion tokens
# dim=100 means each word is a vector of size 100
glove = torchtext.vocab.GloVe(name='6B', dim=100)

print(f"Vector for 'computer' size: {glove['computer'].shape}")
print(f"First 10 values for 'computer': {glove['computer'][:10]}")


## 4. Comparison: GloVe vs FastText

**FastText** is another library (from Facebook) that improves upon Word2Vec.
A key feature of FastText is that it can utilize subword information (character n-grams), though the standard loaded vectors behave similarly to Word2Vec at the word level.

Let's load FastText to compare.



In [ ]:
# Load FastText vectors
# We use 'simple' or 'wikinews' if available in the cache, otherwise it downloads
# Note: 'simple' is smaller than full English FastText
try:
    fasttext = torchtext.vocab.FastText(language='simple')
    print("FastText loaded.")
except Exception as e:
    print("Could not load FastText (might be a connection/download issue). We will focus on GloVe.")
    fasttext = None


## 5. Semantic Similarity and Analogies

The power of embeddings is that **distance = meaning**.
Words with similar meanings are close in the vector space.

### Cosine Similarity
We use Cosine Similarity (angle between vectors) rather than Euclidean Distance, because the magnitude of the vector matters less than its direction.



In [ ]:
import torch.nn.functional as F

def get_cosine_similarity(vec1, vec2):
    # Ensure they are tensors
    v1 = vec1.unsqueeze(0)
    v2 = vec2.unsqueeze(0)
    return F.cosine_similarity(v1, v2).item()

# Compare Synonyms
word1 = "good"
word2 = "great"
word3 = "apple"

sim_1 = get_cosine_similarity(glove[word1], glove[word2])
sim_2 = get_cosine_similarity(glove[word1], glove[word3])

print(f"Similarity {word1} vs {word2}: {sim_1:.4f}")
print(f"Similarity {word1} vs {word3}: {sim_2:.4f}")


### Finding Nearest Neighbors
To find which word is most similar to a given vector, we calculate the similarity against the entire vocabulary.



In [ ]:
def get_nearest_neighbors(embedding_vc, query_word, k=5):
    if query_word not in embedding_vc.stoi:
        return "Word not in vocabulary"
    
    query_vec = embedding_vc[query_word]
    
    # Calculate similarity with ALL vectors in the vocab
    # vectors shape: [vocab_size, dim]
    # query shape: [dim]
    
    # We use matrix multiplication for speed (Dots product approximation of cosine if normalized)
    # Ideally should use full cosine sim, but let's stick to Torchtext's utilities or manual loop
    
    # Using Torch Pdist is heavy for 400k words.
    
    # Let's verify if the embedding object has a helper.
    # Standard GloVe object is a 'Vectors' object.    
    
    # We will compute cosine similarity manually against top N words for speed
    # or just brute force it for demonstration (it accepts matrix batching)
    
    all_dists = F.cosine_similarity(query_vec.unsqueeze(0), embedding_vc.vectors)
    topk_values, topk_indices = torch.topk(all_dists, k+1) # +1 because the word itself is top 1
    
    results = []
    for i in range(1, k+1): # Skip itself
        idx = topk_indices[i].item()
        word = embedding_vc.itos[idx]
        score = topk_values[i].item()
        results.append((word, score))
        
    return results

print(f"Nearest neighbors to 'king' (GloVe): {get_nearest_neighbors(glove, 'king')}")
if fasttext:
    print(f"Nearest neighbors to 'king' (FastText): {get_nearest_neighbors(fasttext, 'king')}")


### Word Analogies
The classic example: `King - Man + Woman = ?`
This demonstrates that the vector space captures **relationships**.



In [ ]:
def solve_analogy(embedding_vc, w1, w2, w3):
    # E.g., King - Man + Woman
    # w1 - w2 + w3
    vec = embedding_vc[w1] - embedding_vc[w2] + embedding_vc[w3]
    
    # Find nearest neighbor to the result vector
    all_dists = F.cosine_similarity(vec.unsqueeze(0), embedding_vc.vectors)
    top_val, top_idx = torch.topk(all_dists, 1)
    
    return embedding_vc.itos[top_idx.item()]

print(f"Analogy: King - Man + Woman = {solve_analogy(glove, 'king', 'man', 'woman')}")
print(f"Analogy: Paris - France + Germany = {solve_analogy(glove, 'paris', 'france', 'germany')}")


## 6. Handling OOV (Out of Vocabulary)

Pre-trained models are fixed. If a user types a word not in the 6 Billion tokens (like a new slang 'rizz'), the model fails.

Common strategies:
1.  **Uniform Random**: Assign a random vector (bad for consistency).
2.  **Mean Vector**: Assign the average of all vectors (safe baseline).
3.  **Zero**: Assign a vector of zeros (ignore it).



In [ ]:
oov_word = "neurlink_xyz" # Intentionally fake

if oov_word in glove.stoi:
    print(f"Found {oov_word}")
else:
    print(f"'{oov_word}' is NOT in vocabulary.")
    
    # Strategy: Mean vector
    mean_vec = torch.mean(glove.vectors, dim=0)
    print(f"Mean vector sample: {mean_vec[:5]}")
